In [1]:
import pandas as pd
import numpy as np
import joblib
import os

# 1. Path to your saved model
# Assuming the notebook is in 'CGPA Project' or the root 'cgpa_predict'
model_path = "best_cgpa_model_v2.pkl" 
if not os.path.exists(model_path):
    # Try alternate path if running from a different directory
    model_path = "CGPA Project/best_cgpa_model_v2.pkl"

print(f"Loading Model from: {model_path} ...")
model = joblib.load(model_path)
print("✅ Final Stacking Ensemble Model Loaded Successfully!\n")

# 2. Define the new student's raw inputs
# You can change these numbers right now to show Sir different scenarios!
student_data = {
    "midterm_norm": 85.0,        # Out of 100
    "assign_norm": 88.0,         # Out of 100
    "twelfth_pct": 92.5,         # 12th Grade %
    "tenth_pct": 90.0,           # 10th Grade %
    "study_hours": 4.5,          # Hours per day
    "attendance": 85.0,          # %
    "backlogs": 0.0,             # Number of backlogs
    "stress": 0.0,               # 0 (Low) or 1 (High)
    "distance": 5.0,             # KM from campus
    "complexity": 2.0,           # 1 (Easy), 2 (Med), 3 (Hard)
    "teacher_fb": 3.0,           # 1 (Poor), 2 (Avg), 3 (Good)
    "participation": 3.0,        # 1-4 scale
    "prev_prev_gpa": 7.8,        # Last semester GPA
    
    # New Modalities added in V2 
    "intro_grade": 8.0,          # Audio Intro Score (1-10)
    "hw_grade": 7.5,             # Handwriting Quality Score (1-10)
}

# 3. Calculate the engineered features just like the pipeline does
student_data["academic_score"] = (student_data["midterm_norm"] + student_data["assign_norm"]) / 2
student_data["school_avg"]     = (student_data["twelfth_pct"] + student_data["tenth_pct"]) / 2
student_data["attend_stress"]  = student_data["attendance"] * (1 - student_data["stress"] * 0.1)
student_data["backlogs_log"]   = np.log1p(student_data["backlogs"])
student_data["has_prev_gpa"]   = 1 if not pd.isna(student_data["prev_prev_gpa"]) else 0

# 4. Convert to a DataFrame (the model expects exactly 20 columns)
# The order must match the FEATURES list in your training script
feature_order = [
    "midterm_norm", "assign_norm", "twelfth_pct", "tenth_pct",
    "study_hours", "attendance", "backlogs", "stress", "distance",
    "complexity", "teacher_fb", "participation", "prev_prev_gpa",
    "academic_score", "school_avg", "attend_stress", "backlogs_log",
    "has_prev_gpa", "intro_grade", "hw_grade"
]

# Create exactly 1 row of data
df_new_student = pd.DataFrame([student_data], columns=feature_order)

# 5. Make the Prediction
predicted_cgpa = model.predict(df_new_student)[0]
predicted_cgpa = np.clip(predicted_cgpa, 0, 10)  # Ensure it doesn't accidentally exceed bounds

# 6. Display the result nicely
print("=" * 50)
print(f"🎓 NEW STUDENT PREDICTION 🎓")
print("=" * 50)
print(f"Midterm & Assignments:  {student_data['academic_score']:.1f}/100")
print(f"High School Average:    {student_data['school_avg']:.1f}%")
print(f"Attendance & Stress:    {student_data['attendance']}% (Stress Level: {student_data['stress']})")
print(f"Audio Intro Quality:    {student_data['intro_grade']}/10")
print(f"Handwriting Neatness:   {student_data['hw_grade']}/10")
print("-" * 50)
print(f"➡️  PREDICTED FINAL CGPA:  {predicted_cgpa:.2f} / 10.00")
print("=" * 50)


Loading Model from: best_cgpa_model_v2.pkl ...
✅ Final Stacking Ensemble Model Loaded Successfully!

🎓 NEW STUDENT PREDICTION 🎓
Midterm & Assignments:  86.5/100
High School Average:    91.2%
Attendance & Stress:    85.0% (Stress Level: 0.0)
Audio Intro Quality:    8.0/10
Handwriting Neatness:   7.5/10
--------------------------------------------------
➡️  PREDICTED FINAL CGPA:  8.43 / 10.00


d:\Code Playground\cgpa_predict\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
